# 4. Cold outreach, built from nothing

The other half of the system. Notebooks 1-3 were all about replying to
someone who already texted back. This is the first message — the one sent
to a stranger who never asked to hear from me.

Different problem, different shape. There's no conversation history, no
question to answer, and nobody to escalate to. Just: given a row from a
scraped CSV, what's the one text most likely to get a reply.

Fakes again, no API key.


## What I'm starting with

A lead row. Scraped business listings, so the fields are inconsistent —
some have hours, some don't, some have 200 reviews and some have 3.


In [ ]:
PROSPECT = {
    'name': 'Yyz Plumbing',
    'primary_type': 'plumber',
    'neighborhood': 'North York',
    'rating': 4.8,
    'review_count': 11,
    'opening_hours': {'monday': '9:00 AM - 5:00 PM', 'saturday': 'Closed'},
}

THIN = {'name': 'Wexford Plumbing', 'phone': '+15551234567'}   # nothing else
print(PROSPECT['name'], '|', THIN['name'])


That second one matters. Only `name` and `phone` are guaranteed, so
anything I build has to degrade gracefully rather than crash or produce a
message with `None` in it.

## Attempt 1: a template


In [ ]:
def draft_v1(p):
    return (f"Hi {p['name']}, we help {p['primary_type']}s in "
            f"{p['neighborhood']} stop missing calls. Interested?")

print(draft_v1(PROSPECT))
try:
    print(draft_v1(THIN))
except KeyError as e:
    print('crashes on thin data:', e)


Two problems. It breaks on thin rows, and every message is identical
except for three nouns — which reads as mail-merge, because it is.

## Attempt 2: one agent, one message


In [ ]:
def fake_llm(prompt, persona='professional'):
    """Stands in for the model. Returns something persona-flavoured."""
    name = prompt.split('BUSINESS:')[1].split('\n')[0].strip()
    bank = {
        'professional': f'Hi {name} - noticed you close at 5pm. VoiceCaptures answers '
                        'the after-hours calls you miss and books them. Worth a look?',
        'witty': f"{name} - your 4.8 stars deserve better than voicemail. We pick up "
                 'when you\'re under a sink.',
        'executive': f'{name}: after-hours calls are going unanswered. We fix that. '
                     'Two minutes to see how?',
    }
    return bank[persona]

def draft_v2(p):
    prompt = f'BUSINESS: {p["name"]}\nWrite a cold SMS.'
    return fake_llm(prompt)

print(draft_v2(PROSPECT))


Handles thin data (the model just says less), and sounds human. But I've
got no idea whether it's any good. One draft, no comparison, no signal.

## Attempt 3: ask for three

Obvious next move — ask the model for three variants and pick one.


In [ ]:
def draft_v3(p):
    prompt = f'BUSINESS: {p["name"]}\nWrite THREE cold SMS variants: professional, witty, executive.'
    # A real model returns all three in one response, having seen the first
    # two while writing the third.
    return [fake_llm(prompt, k) for k in ('professional', 'witty', 'executive')]

for d in draft_v3(PROSPECT):
    print('-', d)


This is where I'd have stopped if I weren't paying attention. It looks
the same as what I ended up with, and it isn't.

One model call producing three variants means the model sees its own
earlier drafts while writing the later ones. They converge — variant three
is a rewrite of variant one wearing a different hat. You get three drafts
with one draft's worth of diversity.

Three separate calls, in parallel, each blind to the others, actually
explore three different spaces.


In [ ]:
import asyncio

PERSONAS = {
    'professional': 'Direct and credible. No jokes. Lead with the business problem.',
    'witty': 'Light and human. One bit of personality, never a pun for its own sake.',
    'executive': 'Blunt. Under 20 words if possible. Assumes their time is expensive.',
}

async def draft_one(prospect, persona, angle):
    await asyncio.sleep(0)      # pretend network call
    prompt = f'BUSINESS: {prospect["name"]}\nANGLE: {angle}\nSTYLE: {PERSONAS[persona]}'
    return persona, fake_llm(prompt, persona)

async def draft_all(prospect, angle):
    results = await asyncio.gather(*(draft_one(prospect, k, angle) for k in PERSONAS))
    return dict(results)

drafts = await draft_all(PROSPECT, 'closes at 5pm, misses after-hours calls')
for k, v in drafts.items():
    print(f'{k:14} {v}')


Parallel matters practically too: three sequential calls is three round
trips. `asyncio.gather` makes it one wall-clock wait.

## The angle problem

Notice I passed an `angle` in. That's not incidental — it was the fix for
something that made the picker useless.

Without it, each drafting agent picks its own personalization. So the
professional one talks about after-hours calls, the witty one talks about
the star rating, the executive one talks about review count. Then the
picker has to compare three different *strategies* while also judging
three different writing styles, and I can't tell which dimension it
actually chose on.


In [ ]:
def fake_hook_agent(p):
    """Picks ONE angle, grounded in one concrete fact. Never invents."""
    hours = p.get('opening_hours') or {}
    if any('5:00 PM' in str(v) or 'Closed' in str(v) for v in hours.values()):
        return {'angle': 'closes early / weekends, so after-hours calls go unanswered',
                'supporting_fact': f"hours: {hours}"}
    if p.get('rating', 0) >= 4.5 and p.get('review_count', 999) < 25:
        return {'angle': 'well rated but few reviews - calls not converting to jobs',
                'supporting_fact': f"{p['rating']} stars from {p['review_count']} reviews"}
    return {'angle': 'home service work is urgent, missed calls are lost jobs',
            'supporting_fact': f"business type: {p.get('primary_type', 'home services')}"}

print(fake_hook_agent(PROSPECT))
print(fake_hook_agent(THIN))     # degrades to the generic angle, no crash


The thin row falls through to a generic-but-relevant angle. No
special-casing needed — the fallback is just the last branch.

`supporting_fact` exists so the angle is traceable to something real in
the data. It's the same instinct as the grounding guardrail: if the
personalization can't point at a fact, it's invention.

Now all three drafts anchor on the same angle, and the picker is judging
one variable.

## The picker, and the mistake I avoided


In [ ]:
def fake_picker_naive(drafts):
    """Returns the winning TEXT. This is the tempting version."""
    winner = drafts['professional']
    return winner.replace('Worth a look?', 'Worth a quick look?')   # 'improving' it

chosen = fake_picker_naive(drafts)
print('picker returned:', chosen)
print()
print('matches any original draft:', chosen in drafts.values())


The picker rewrote it while selecting. Subtly — one word — and now the
message going out isn't what any drafting agent wrote.

That breaks two things. I can't guarantee the sent text was the vetted
text, and I can't attribute a reply to a persona, because the thing that
got sent is a fourth variant nobody evaluated.

Fix: the picker returns a **label**, and code looks up the original.


In [ ]:
def fake_picker(drafts):
    """Structured output: which persona won, and one phrase on why."""
    return {'winner': 'professional',
            'reason': 'clear, concise, names the pain and the action'}

decision = fake_picker(drafts)
chosen = drafts[decision['winner']]        # code does the lookup
print(f"winner: {decision['winner']} ({decision['reason']})")
print('exactly as drafted:', chosen in drafts.values())
print(chosen)


Same trick as taking the send tool off the SDR agent, one layer up: the
agent returns a *decision*, code performs the action. The agent can't
quietly do something on the way past.

The persona label is also a free experiment. Store it on every outbound
message and reply rate by persona falls out of the data — a one-shot
picker turns into something that tells me which style actually works.

## Compliance is not the agent's job

Canadian cold SMS needs an opt-out. My first instinct was to put it in the
drafting instructions.


In [ ]:
COMPLIANCE_FOOTER = 'Reply YES for a demo, NO to opt-out.'

def draft_with_instruction(persona):
    # 'Always end with the opt-out line' - a request, not a guarantee.
    text = drafts[persona]
    forgot = persona == 'witty'         # models drop instructions sometimes
    return text if forgot else f'{text} {COMPLIANCE_FOOTER}'

for k in drafts:
    out = draft_with_instruction(k)
    print(f"{k:14} has opt-out: {COMPLIANCE_FOOTER in out}")


One missing. In testing that's a shrug; in production it's a message to a
real Canadian business with no opt-out on it.

Anything with legal weight shouldn't be something I *ask* a model to
remember. Append it in code, after the agent is done.


In [ ]:
def finalize(text):
    if COMPLIANCE_FOOTER not in text:
        text = f'{text} {COMPLIANCE_FOOTER}'
    return text

for k, v in drafts.items():
    print(f'{k:14} {COMPLIANCE_FOOTER in finalize(v)}')


## The whole pipeline


In [ ]:
SENT = []

def send_sms(to, body):
    SENT.append((to, body))
    return 'delivered'

async def run_cold_outreach(prospect, dry_run=False):
    hook = fake_hook_agent(prospect)
    drafts = await draft_all(prospect, hook['angle'])
    decision = fake_picker(drafts)
    text = finalize(drafts[decision['winner']])

    result = {'angle': hook['angle'], 'supporting_fact': hook['supporting_fact'],
              'winner': decision['winner'], 'reason': decision['reason'],
              'sent_text': text, 'all_drafts': drafts}

    if dry_run:
        return result                      # what the admin Generate button uses

    send_sms(prospect.get('phone', '+15550000000'), text)   # plain call, not a tool
    return result

r = await run_cold_outreach(PROSPECT, dry_run=True)
for k in ('angle', 'supporting_fact', 'winner', 'reason'):
    print(f'{k:18} {r[k]}')
print(f"\n{r['sent_text']}")


`dry_run` is what the admin console's **Generate suggested reply** button
calls on a prospect with no history — runs the whole pipeline, returns the
text and the reasoning, sends nothing.

## Why this path has no guardrails

Worth stating, because the asymmetry with the reply path looks like an
oversight and isn't.

The autopilot reply path has two guardrails and a human fallback. This
one has none, and runs fully autonomously.

The difference is what a bad output costs. A weak first message gets
ignored — the downside is a wasted send. A wrong *reply* gets quoted back
at me, because by then it's a conversation and the prospect is paying
attention.

And the risky material is already excluded by construction: the drafting
agents get a name, a type, an angle and a fact. No pricing, no contract
terms, no KB. They can't state a price they were never given.

The one thing that *does* have legal weight — the opt-out — is appended
in code precisely because it can't be left to the agent.

If I added product claims to the cold message, this reasoning stops
holding and it'd need the grounding check too.

## What this became

| here | in the repo |
|---|---|
| `fake_hook_agent` | `app/agents/hook_agent.py`, returns `Hook(angle, supporting_fact)` |
| `PERSONAS` + `draft_one` | `app/agents/drafting_agents.py`, three `Agent` objects |
| `draft_all` | the `asyncio.gather` in `cold_outreach.py` |
| `fake_picker` | `app/agents/picker_agent.py`, returns `PickerDecision(winner, reason)` |
| `finalize` | compliance footer appended in code |
| `run_cold_outreach` | `run_cold_outreach_for_prospect` in `app/agents/cold_outreach.py` |
| `dry_run` | same flag, used by `/admin` and `test_send.py` |

Look at the real personas and the picker's decision type:


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else '.')
for k, v in {'SUPABASE_URL':'https://t.supabase.co','SUPABASE_SERVICE_KEY':'t',
             'OPENAI_API_KEY':'sk-t','TWILIO_ACCOUNT_SID':'ACt',
             'TWILIO_AUTH_TOKEN':'t','TWILIO_FROM_NUMBER':'+15550000000'}.items():
    os.environ.setdefault(k, v)

from app.agents.drafting_agents import DRAFTING_AGENTS
from app.agents.picker_agent import PickerDecision
from app.agents.hook_agent import Hook

print('personas:', list(DRAFTING_AGENTS))
print('picker returns:', list(PickerDecision.model_fields))
print('hook returns  :', list(Hook.model_fields))


## Still open on this path

- **No outcome data.** I track which persona won every pick, but I've
  never joined that against reply rates. The experiment is instrumented
  and unread. That's the single highest-value thing left here.
- **The picker is unvalidated.** It's a model guessing what a plumber
  would reply to. Whether it agrees with reality is exactly what reply
  rate by persona would tell me.
- **Three personas is arbitrary.** I picked them because they felt
  different. No evidence three is better than two or five.
